# Figure 3J: representative RNA expression on the embryonic UMAP

Read the original accepted `tree_results.pkl` and `Gene expression plots.h5`. The source gene list, UMAP coordinates, expression values, colors and drawing settings are retained. If the compact plotting H5 is absent, the original packaging branch uses the annotated intermediate inputs below.

Replace each `ROOT` placeholder with the required file or directory path before running. This file includes its plotting and editable-PDF exporter; it does not execute another notebook or a private validation script.


In [ ]:
#1. =========== Shared configuration and imports
from pathlib import Path
import os, sys, json, pickle, hashlib, warnings, subprocess

# Replace each ROOT placeholder with the required file or directory path before running.
ROOT = Path(".")
# Set the file or directory path.
TREE_DIR = ROOT
# Set the file path. Uses tree_results.pkl generated by the original tree analysis in 04_tree.ipynb.
BUNDLE_FILE = ROOT
# Set the file path. Uses Gene expression plots.h5 generated by the result-packaging step of 04_Gene_expression_plots.ipynb.
RESULT_FILE = ROOT
# Set the file or directory path.
OUTPUT_DIR = ROOT
# Set the file or directory path.
PNG_FILE = ROOT
# Set the file or directory path.
PDF_FILE = ROOT
# Set the file or directory path.
WORK = ROOT
# Set the file or directory path.
RUNTIME = ROOT
for folder in (OUTPUT_DIR, RUNTIME):
    folder.mkdir(parents=True, exist_ok=True)
# Set the file or directory path.
os.environ['MPLCONFIGDIR'] = str(ROOT)
os.environ['TMP'] = str(RUNTIME)
os.environ['TEMP'] = str(RUNTIME)
sys.dont_write_bytecode = True

import numpy as np
import pandas as pd
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager, colors
from matplotlib.collections import PathCollection, QuadMesh, LineCollection
from matplotlib.markers import MarkerStyle
from PIL import Image, ImageChops

try:
    import reportlab, pypdf
except ModuleNotFoundError:
    # Set the file or directory path.
    bundled_packages = ROOT
    if not bundled_packages.is_dir():
        raise RuntimeError('Install reportlab and pypdf in the active environment.')
    sys.path.append(str(bundled_packages))
    import reportlab, pypdf
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

GENES = ['Neurod1', 'Insm1', 'Gata3', 'Myt1l', 'Mafb', 'Prox1', 'Thrb', 'Islr2',
         'Lypd1', 'Dio3', 'Tshz3', 'Calb2', 'Etv4', 'Mgat4c', 'Tac1']

def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def array_sha(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()

tree_sha_before = file_sha(BUNDLE_FILE)
result_existed = RESULT_FILE.exists()
# Set the file path. Uses Fig3J.png generated by the original accepted Fig3J rendering step.
accepted_preview = ROOT


In [ ]:
#2. =========== Load the fixed tree and the accepted plotting data
with warnings.catch_warnings(), BUNDLE_FILE.open('rb') as stream:
    warnings.simplefilter('ignore', DeprecationWarning)
    bundle = pickle.load(stream)
obs = bundle['obs']
order = bundle['plot_order']
projection = bundle['projection']
assert len(obs) == 7355
assert hashlib.sha256(bundle['model']['F'].tobytes() + bundle['model']['B'].tobytes()).hexdigest() == bundle['model_hash']
assert array_sha(bundle['plot_coordinates']) == bundle['coordinates_hash']
assert np.array_equal(np.sort(order), np.arange(len(obs)))

if result_existed:
    result_sha_before = file_sha(RESULT_FILE)
    with h5py.File(RESULT_FILE, 'r') as handle:
        assert handle.attrs['schema'] == 'Gene_expression_plots_v1'
        assert handle.attrs['tree_bundle_sha256'] == tree_sha_before
        assert handle.attrs['tree_model_hash'] == bundle['model_hash']
        assert np.array_equal(handle['cell_ids'].asstr()[()], obs.index.to_numpy(dtype=str))
        assert list(handle['gene_symbols'].asstr()[()]) == GENES
        xy = handle['X_umap'][()]
        expression = handle['expression'][()]
        gene_ids = handle['gene_ids'].asstr()[()]
        segment_labels = handle['seg'].asstr()[()]
        library_total = handle['library_total'][()]
        scale = str(handle.attrs['expression_calculation'])
        provenance = json.loads(handle.attrs['provenance_json'])
        saved_lut = handle['color_lut'][()]
        assert np.array_equal(handle['plot_order'][()], order)
        assert np.array_equal(handle['t'][()], projection['t'])
        for key in obs.columns:
            stored = handle['obs'][key]
            if stored.dtype.kind in 'OSU':
                assert np.array_equal(stored.asstr()[()], obs[key].astype(str).to_numpy())
            else:
                np.testing.assert_equal(stored[()], obs[key].to_numpy())
        assert array_sha(xy) == handle.attrs['umap_raw_bytes_sha256']
        assert array_sha(expression) == handle.attrs['expression_raw_bytes_sha256']
        assert np.array_equal(handle['gene_maxima'][()], expression.max(axis=0))
    print('Loaded the saved tree and one minimal result H5; no temporary inputs required.')
else:
    # First creation only: copy accepted values without any new fitting or normalization.
    # Set the file path. Uses expression.h5 generated by the original trajectory-expression input preparation step.
    EXPRESSION_FILE = ROOT
    # Set the file path. Uses cells.tsv generated by the original trajectory-expression input preparation step.
    CELLS_FILE = ROOT
    # Set the file path. Uses input_meta.json generated by the original trajectory-expression input preparation step.
    INPUT_META = ROOT
    # Set the file path. Uses CSS_3000_Embryonic_Inte_adata.h5ad generated by fig1_Embryonic_integration.ipynb.
    EMBEDDING_FILE = ROOT
    protected = [BUNDLE_FILE, EXPRESSION_FILE, CELLS_FILE, INPUT_META, EMBEDDING_FILE]
    before = {str(path): file_sha(path) for path in protected}
    with warnings.catch_warnings(), BUNDLE_FILE.open('rb') as stream:
        warnings.simplefilter('ignore', DeprecationWarning)
        bundle = pickle.load(stream)
    obs = bundle['obs']
    tree_xy = bundle['plot_coordinates']
    order = bundle['plot_order']
    projection = bundle['projection']
    assert bundle['schema_version'] == 1
    assert tree_xy.shape == (7355, 2)
    assert np.array_equal(tree_xy[:, 1], projection['t'])
    assert np.array_equal(np.sort(order), np.arange(len(obs)))
    assert hashlib.sha256(tree_xy.tobytes()).hexdigest() == bundle['coordinates_hash']
    assert hashlib.sha256(bundle['model']['F'].tobytes() + bundle['model']['B'].tobytes()).hexdigest() == bundle['model_hash']
    meta = json.loads(INPUT_META.read_text(encoding='utf-8'))
    assert meta['tree_model_hash'] == bundle['model_hash']
    assert meta['tree_coordinates_hash'] == bundle['coordinates_hash']
    assert before[str(EMBEDDING_FILE)] == bundle['fitted_source_sha256']
    with h5py.File(EMBEDDING_FILE, 'r') as handle:
        assert np.array_equal(handle['obs/_index'].asstr()[()], obs.index.to_numpy(dtype=str))
        xy = handle['obsm/X_umap'][()]
        for key in ['cell_type_plot', 'curated_cell_type']:
            value = handle['obs'][key]
            if isinstance(value, h5py.Group) and 'categories' in value:
                labels = value['categories'].asstr()[()][value['codes'][()]]
            elif isinstance(value, h5py.Group) and 'values' in value:
                assert not value['mask'][()].any()
                labels = value['values'].asstr()[()]
            else:
                labels = value.asstr()[()]
            assert np.array_equal(labels.astype(str), obs['state'].astype(str))
    assert xy.shape == (7355, 2) and np.isfinite(xy).all()
    umap_hash = hashlib.sha256(xy.tobytes()).hexdigest()

    with h5py.File(EXPRESSION_FILE, 'r') as handle:
        assert np.array_equal(handle['cell_ids'].asstr()[()], obs.index.to_numpy(dtype=str))
        symbols = pd.Index(handle['gene_symbols'].asstr()[()])
        positions = symbols.get_indexer(GENES)
        assert np.all(positions >= 0)
        column_order = np.argsort(positions)
        expression = handle['expression'][:, positions[column_order]][:, np.argsort(column_order)]
        scale = str(handle.attrs['calculation'])
    assert expression.shape == (7355, 15) and np.isfinite(expression).all()
    assert np.all(expression >= 0)
    cells = pd.read_csv(CELLS_FILE, sep='\t', index_col=0)
    assert cells.index.equals(obs.index)
    for key in ['state', 'stage_standardized', 'study_id', 'library_id']:
        assert np.array_equal(cells[key].astype(str), obs[key].astype(str))


    with h5py.File(EXPRESSION_FILE, 'r') as handle:
        gene_ids = handle['gene_ids'].asstr()[()][positions]
        library_total = handle['library_total'][()]
    segment_labels = cells['seg'].astype(str).to_numpy()
    provenance = {
        'sources_sha256': {str(Path(path).relative_to(ROOT)): sha for path, sha in before.items()},
        'expression_provenance': meta,
        'accepted_preview_sha256': file_sha(accepted_preview),
        'coordinate_system': 'Existing Fig1/Fig2 UMAP; not rectangular tree display',
        # Set the file path. Uses tree_results.pkl generated by the original tree analysis in 04_tree.ipynb.
        'normal_redraw_inputs': [str(ROOT),
                               # Set the file path. Uses Gene expression plots.h5 generated by the result-packaging step of 04_Gene_expression_plots.ipynb.
                               str(ROOT)],
    }
    assert before == {str(path): file_sha(path) for path in protected}
    print('Copied the accepted 15 expression columns and original UMAP for one-time packaging.')

assert xy.shape == (7355, 2) and xy.dtype == np.float32
assert expression.shape == (7355, 15) and expression.dtype == np.float64
assert np.isfinite(xy).all() and np.isfinite(expression).all() and (expression >= 0).all()
assert (library_total > 0).all()
umap_hash = array_sha(xy)
assert umap_hash == '72a0ad1c21b54296f0fc9e33bbb64b2a727d9d08c0bc2a56c43ba63f0cfa1572'


In [ ]:
#3. =========== Accepted appearance and measured-expression rendering
FIGSIZE = (20.0, 6.0)       # Two rows: eight genes, then seven and the color key.
DPI = 600                  # PNG resolution; PDF uses editable vectors.
POINT_SIZE = 1.35           # Point area only; stored UMAP is never changed.
POINT_ALPHA = 1.0           # Opaque points prevent density-dependent alpha blending.
ZERO_COLOR = '#B0B0B0'      # Measured zero is separate from low positive expression.
COLOR_EDGE_GAMMA = 1.8     # >1 slows the center and accelerates both color-scale ends.
base_colors = colors.LinearSegmentedColormap.from_list(
    'muted_blue_red', ['#275D7D', '#85A7B9', '#C2CDD0', '#C99691', '#BC5B58', '#D71920'], N=1024)
color_fraction = np.linspace(0, 1, 1024)
color_position = 0.5 + 0.5 * np.sign(2 * color_fraction - 1) * np.abs(2 * color_fraction - 1) ** COLOR_EDGE_GAMMA
assert np.all(np.diff(color_position) >= 0)
np.testing.assert_allclose(color_position[[0, -1]], [0, 1])
assert color_position[102] - color_position[0] > color_position[563] - color_position[461]
COLORMAP = colors.ListedColormap(base_colors(color_position), name='edge_emphasis_blue_red')
TITLE_SIZE = 12

# Set the file or directory path.
font_dir = ROOT
for font_file in ['arial.ttf', 'ariali.ttf', 'arialbd.ttf']:
    font_manager.fontManager.addfont(str(font_dir / font_file))
plt.rcParams.update({'font.family': 'Arial', 'font.size': 9,
                     'axes.linewidth': 0.6, 'savefig.facecolor': 'white'})
fig, axes = plt.subplots(2, 8, figsize=FIGSIZE)
fig.subplots_adjust(left=0.012, right=0.99, bottom=0.07, top=0.88,
                    hspace=0.22, wspace=0.08)
titles = []
maxima = expression.max(axis=0)
center = (xy.min(axis=0) + xy.max(axis=0)) / 2
radius = float(np.ptp(xy, axis=0).max()) * 0.54

for index, (gene, ax) in enumerate(zip(GENES, axes.flat)):
    values = expression[:, index]
    relative = values / maxima[index] if maxima[index] else np.zeros(len(obs))
    assert np.all((relative >= 0) & (relative <= 1))
    draw_order = order[np.argsort(values[order], kind='stable')]
    point_colors = COLORMAP(relative)
    point_colors[values == 0] = colors.to_rgba(ZERO_COLOR)
    points = ax.scatter(xy[draw_order, 0], xy[draw_order, 1], c=point_colors[draw_order],
                        s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0, zorder=2)
    np.testing.assert_array_equal(np.sort(draw_order), np.arange(len(obs)))
    np.testing.assert_array_equal(points.get_offsets(), xy[draw_order])
    np.testing.assert_array_equal(points.get_facecolors(), point_colors[draw_order])
    titles.append(ax.set_title(gene, fontsize=TITLE_SIZE, fontstyle='italic',
                               fontweight='normal', pad=5))
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_aspect('equal', adjustable='box')
    ax.set_axis_off()

fig.suptitle('Fig3j preview | existing Fig1/Fig2 UMAP coordinates',
             x=0.5, y=0.98, fontsize=13)
legend_ax = axes.flat[-1]
legend_ax.set_axis_off()
legend_ax.text(0.5, 0.90, 'expression', transform=legend_ax.transAxes,
               ha='center', va='center', fontsize=11)
legend_box = legend_ax.get_position()
cax = fig.add_axes([legend_box.x0 + legend_box.width * 0.19,
                    legend_box.y0 + legend_box.height * 0.69,
                    legend_box.width * 0.62, 0.015])
colorbar = fig.colorbar(matplotlib.cm.ScalarMappable(norm=colors.Normalize(0, 1), cmap=COLORMAP),
                       cax=cax, orientation='horizontal')
colorbar.set_ticks([0, 0.5, 1])
colorbar.set_ticklabels(['0+', '0.5 max', 'max'])
colorbar.ax.tick_params(labelsize=7, length=2, width=0.5)
colorbar.outline.set_visible(False)
legend_ax.text(0.5, 0.43, 'log10(1 + CPM)\nper-gene max; nonlinear colors',
               transform=legend_ax.transAxes, ha='center', fontsize=9, linespacing=1.4)
legend_ax.scatter([0.20], [0.23], s=18, color=ZERO_COLOR,
                  transform=legend_ax.transAxes, linewidths=0)
legend_ax.text(0.30, 0.23, 'Not detected (=0)', transform=legend_ax.transAxes,
               ha='left', va='center', fontsize=8)
legend_ax.text(0.5, 0.07, 'White background: no cell', transform=legend_ax.transAxes,
               ha='center', va='center', fontsize=8)
fig.text(0.5, 0.025,
         '7,355 unchanged cells and annotations; existing UMAP loaded without recalculation. '
         f'Measured RNA, no imputation. Nonlinear color mapping (gamma = {COLOR_EDGE_GAMMA}); higher expression drawn last.',
         ha='center', va='center', fontsize=8)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for text in titles:
    box = text.get_window_extent(renderer)
    assert 0 <= box.x0 < box.x1 <= fig.bbox.width
    assert 0 <= box.y0 < box.y1 <= fig.bbox.height




In [ ]:
#4. =========== Save one minimal, lossless plotting-result file
parameters = {
    'genes': GENES, 'figsize_inches': list(FIGSIZE), 'dpi': DPI,
    'point_size_pt2': POINT_SIZE, 'point_alpha': POINT_ALPHA, 'zero_color': ZERO_COLOR,
    'color_edge_gamma': COLOR_EDGE_GAMMA,
    'color_anchors': ['#275D7D', '#85A7B9', '#C2CDD0', '#C99691', '#BC5B58', '#D71920'],
    'color_lut_size': len(COLORMAP.colors), 'normalization': 'Divide each log expression by its gene maximum',
    'draw_order': 'plot_order[np.argsort(expression[plot_order], kind="stable")]',
    'title_size_pt': TITLE_SIZE, 'subplot_adjust': {'left': .012, 'right': .99, 'bottom': .07,
        'top': .88, 'hspace': .22, 'wspace': .08},
    'pdf_expected_clips': 0, 'pdf_text': 'Live embedded Arial / Arial-Italic',
    'pdf_graphics': 'Independent closed vector points; vector colorbar rectangles; no raster objects',
}
if not result_existed:
    string_type = h5py.string_dtype('utf-8')
    with h5py.File(RESULT_FILE, 'x') as handle:
        handle.attrs['schema'] = 'Gene_expression_plots_v1'
        handle.attrs['expression_calculation'] = scale
        handle.attrs['tree_bundle_sha256'] = tree_sha_before
        handle.attrs['tree_model_hash'] = bundle['model_hash']
        handle.attrs['tree_coordinates_hash'] = bundle['coordinates_hash']
        handle.attrs['umap_raw_bytes_sha256'] = array_sha(xy)
        handle.attrs['expression_raw_bytes_sha256'] = array_sha(expression)
        handle.attrs['parameters_json'] = json.dumps(parameters, ensure_ascii=False)
        handle.attrs['provenance_json'] = json.dumps(provenance, ensure_ascii=False)
        for key, value in {'expression': expression, 'X_umap': xy, 'plot_order': order,
                           'library_total': library_total, 'gene_maxima': expression.max(axis=0),
                           't': projection['t'], 'color_lut': np.asarray(COLORMAP.colors)}.items():
            handle.create_dataset(key, data=value, compression='gzip', shuffle=True)
        for key, value in {'cell_ids': obs.index.to_numpy(dtype=str),
                           'gene_symbols': GENES, 'gene_ids': gene_ids,
                           'seg': segment_labels}.items():
            handle.create_dataset(key, data=np.asarray(value, dtype=object), dtype=string_type,
                                  compression='gzip')
        group = handle.create_group('obs')
        for key in obs.columns:
            value = obs[key].to_numpy()
            if value.dtype.kind in 'OSU' or isinstance(obs[key].dtype, pd.CategoricalDtype):
                group.create_dataset(key, data=obs[key].astype(str).to_numpy(dtype=object),
                                     dtype=string_type, compression='gzip')
            else:
                group.create_dataset(key, data=value, compression='gzip', shuffle=True)
    with h5py.File(RESULT_FILE, 'r') as handle:
        np.testing.assert_array_equal(handle['expression'][()], expression)
        np.testing.assert_array_equal(handle['X_umap'][()], xy)
        np.testing.assert_array_equal(handle['plot_order'][()], order)
        np.testing.assert_array_equal(handle['library_total'][()], library_total)
        np.testing.assert_array_equal(handle['color_lut'][()], COLORMAP.colors)
        assert np.array_equal(handle['cell_ids'].asstr()[()], obs.index.to_numpy(dtype=str))
    print('Saved:', RESULT_FILE, f'({RESULT_FILE.stat().st_size:,} bytes)')
else:
    assert np.array_equal(saved_lut, np.asarray(COLORMAP.colors))
    assert result_sha_before == file_sha(RESULT_FILE)
    print('Reused the minimal result H5 without modifying it.')


In [ ]:
#5. =========== Export editable PDF from the same finalized Matplotlib canvas
# No rasterization, Form XObjects, shading, or clipping masks are used.
# Set the file or directory path.
pdfmetrics.registerFont(TTFont('GeneArial', str(ROOT)))
# Set the file or directory path.
pdfmetrics.registerFont(TTFont('GeneArialItalic', str(ROOT)))

def closed_circle(pdf, x, y, radius, rgb):
    pdf.setFillColorRGB(*map(float, rgb[:3]))
    path = pdf.beginPath()
    path.circle(float(x), float(y), float(radius))
    path.close()
    pdf.drawPath(path, stroke=0, fill=1)

def polygon(pdf, vertices, rgb):
    pdf.setFillColorRGB(*map(float, rgb[:3]))
    path = pdf.beginPath()
    path.moveTo(*map(float, vertices[0]))
    for vertex in vertices[1:]:
        path.lineTo(*map(float, vertex))
    path.close()
    pdf.drawPath(path, stroke=0, fill=1)

def live_text(pdf, artist, renderer, unit):
    if not artist.get_visible() or not artist.get_text():
        return
    assert float(artist.get_rotation()) == 0
    origin = artist.get_transform().transform(artist.get_unitless_position())
    _, lines, _ = artist._get_layout(renderer)
    font = 'GeneArialItalic' if artist.get_fontstyle() == 'italic' else 'GeneArial'
    text = pdf.beginText()
    text.setFont(font, artist.get_fontsize())
    text.setFillColorRGB(*colors.to_rgba(artist.get_color())[:3])
    for line, _, x, y in lines:
        text.setTextTransform(1, 0, 0, 1, (origin[0] + x) * unit, (origin[1] + y) * unit)
        text.textOut(line)
    pdf.drawText(text)

def editable_pdf(fig, destination):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    unit = 72.0 / fig.dpi
    pdf = canvas.Canvas(str(destination), pagesize=tuple(fig.get_size_inches() * 72),
                        pageCompression=1, invariant=1, initialFontName='GeneArial')
    pdf.setTitle('Gene expression plots')
    pdf.setCreator('04_Gene_expression_plots.ipynb; editable vector export')
    pdf.setFillColorRGB(1, 1, 1)
    pdf.rect(0, 0, *tuple(fig.get_size_inches() * 72), fill=1, stroke=0)
    point_count, colorbar_cells = 0, 0
    for ax in fig.axes:
        for collection in ax.collections:
            if not collection.get_visible():
                continue
            if isinstance(collection, PathCollection):
                offsets = collection.get_offset_transform().transform(collection.get_offsets()) * unit
                sizes = np.resize(collection.get_sizes(), len(offsets))
                facecolors = np.resize(collection.get_facecolors(), (len(offsets), 4))
                assert np.all(facecolors[:, 3] == 1)
                for point, size, rgba in zip(offsets, sizes, facecolors):
                    closed_circle(pdf, *point, np.sqrt(size) / 2, rgba)
                point_count += len(offsets)
            elif isinstance(collection, QuadMesh):
                for path, rgba in zip(collection.get_paths(), collection.get_facecolors()):
                    polygon(pdf, collection.get_transform().transform(path.vertices) * unit, rgba)
                    colorbar_cells += 1
            elif isinstance(collection, LineCollection):
                segments = collection.get_segments()
                for index, segment in enumerate(segments):
                    rgba = collection.get_colors()[index % len(collection.get_colors())]
                    pdf.setStrokeColorRGB(*map(float, rgba[:3]))
                    widths = collection.get_linewidths()
                    pdf.setLineWidth(float(widths[index % len(widths)]))
                    vertices = collection.get_transform().transform(segment) * unit
                    path = pdf.beginPath()
                    path.moveTo(*map(float, vertices[0]))
                    for vertex in vertices[1:]:
                        path.lineTo(*map(float, vertex))
                    pdf.drawPath(path, stroke=1, fill=0)
            else:
                raise TypeError(f'Unsupported collection: {type(collection).__name__}')
    # Only the colorbar axis is visible; expression axes have no ticks or frame.
    for tick in cax.xaxis.get_major_ticks():
        line = tick.tick1line
        if line.get_visible():
            marker = MarkerStyle(line.get_marker())
            path = marker.get_path().transformed(marker.get_transform())
            anchor = line.get_transform().transform(line.get_xydata())[0] * unit
            vertices = path.vertices * line.get_markersize() + anchor
            pdf.setStrokeColorRGB(*colors.to_rgba(line.get_markeredgecolor())[:3])
            pdf.setLineWidth(line.get_markeredgewidth())
            pdf.line(*map(float, vertices[0]), *map(float, vertices[-1]))
    text_artists = list(fig.texts)
    for ax in fig.axes:
        text_artists.extend([ax.title, *ax.texts])
    text_artists.extend(cax.get_xticklabels())
    for artist in text_artists:
        live_text(pdf, artist, renderer, unit)
    assert point_count == 7355 * 15 + 1
    assert colorbar_cells == len(COLORMAP.colors)
    pdf.showPage()
    pdf.save()
    print('Editable PDF:', point_count, 'independent vector circles;', colorbar_cells,
          'vector colorbar cells; live Arial text; 0 clipping groups.')

fig.savefig(PNG_FILE, dpi=DPI)
editable_pdf(fig, PDF_FILE)
plt.close(fig)
with Image.open(PNG_FILE) as image:
    assert image.size == (12000, 3600)
    assert all(abs(value - DPI) < 0.1 for value in image.info['dpi'])
if accepted_preview.exists():
    with Image.open(accepted_preview) as accepted, Image.open(PNG_FILE) as saved:
        assert ImageChops.difference(accepted.convert('RGB'), saved.convert('RGB')).getbbox() is None
    print('The final PNG is pixel-identical to the accepted preview.')
assert file_sha(BUNDLE_FILE) == tree_sha_before
print('Fixed cells, annotations, measured expression, UMAP, and tree verified unchanged.')


## Inputs and reuse

Use `tree_results.pkl` from the original tree analysis and `Gene expression plots.h5` from the original Fig3J result-packaging step. For a first packaging run, set the expression, cell-table, metadata and integration inputs in the corresponding branch. The tree and saved result checks, measured-zero handling, selected 15 genes and editable export are unchanged.



# Existing measured-expression workflow
The original panel code above already contains its existing measured-expression
loading, checks, rendering and compact-result handling. No new statistical or
model-fitting stage is introduced for this panel.
